# ARCH and GARCH Volatility Models

Module: Financial Time Series

## Lesson summary

This notebook introduces ARCH and GARCH volatility models as tools for describing volatility clustering in financial returns.

## Learning objectives
- Explain why conditional variance matters in financial time series.
- Fit an ARCH/GARCH model to asset returns.
- Interpret model output and volatility forecasts.
- Connect volatility modeling with risk measurement.

## Lesson flow
1. Review ARCH and GARCH concepts.
2. Build a reproducible return panel.
3. Fit volatility models with `arch_model`.
4. Inspect conditional volatility and model diagnostics.


## Setup

Start from a clean return series with a date index. The notebook may use live or previously downloaded prices, so record the source and date range before fitting volatility models.


## ARCH (q)
Autoregressive conditional heteroskedasticity

In econometrics, the autoregressive conditional heteroskedasticity (ARCH) model is a statistical model for time series data that describes the variance of the current error term or innovation as a function of the actual sizes of the previous time periods' error terms;often the variance is related to the squares of the previous innovations. The ARCH model is appropriate when the error variance in a time series follows an autoregressive (AR) model; if an autoregressive moving average (ARMA) model is assumed for the error variance, the model is a generalized autoregressive conditional heteroskedasticity (GARCH) model.

ARCH models are commonly employed in modeling financial time series that exhibit time-varying volatility and volatility clustering, i.e. periods of swings interspersed with periods of relative calm. ARCH-type models are sometimes considered to be in the family of stochastic volatility models, although this is strictly incorrect since at time t the volatility is completely pre-determined (deterministic) given previous values.



$$ 
\sigma_t^{2}=\omega+\sum_{i=1}^{p}\alpha_{i}\epsilon_{t-i}^{2} 
$$ 

## GARCH (p,q)
Generalized autoregressive conditional heteroskedasticity

In econometrics, the autoregressive conditional heteroskedasticity (ARCH) model is a statistical model for time series data that describes the variance of the current error term or innovation as a function of the actual sizes of the previous time periods' error terms; often the variance is related to the squares of the previous innovations. The ARCH model is appropriate when the error variance in a time series follows an autoregressive (AR) model; if an autoregressive moving average (ARMA) model is assumed for the error variance, the model is a generalized autoregressive conditional heteroskedasticity (GARCH) model.

ARCH models are commonly employed in modeling financial time series that exhibit time-varying volatility and volatility clustering, i.e. periods of swings interspersed with periods of relative calm. ARCH-type models are sometimes considered to be in the family of stochastic volatility models, although this is strictly incorrect since at time t the volatility is completely pre-determined (deterministic) given previous values.

In this class of processes, the variance dynamics are

$$
\sigma_{t}^{2}=\omega
+ \sum_{i=1}^{p}\alpha_{i}\epsilon_{t-i}^{2}
+\sum_{k=1}^{q}\beta_{k}\sigma_{t-k}^{2}
$$ 

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

# Visualization 
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn

seaborn.set_style("darkgrid")
plt.rc("figure", figsize=(16, 6))
plt.rc("savefig", dpi=90)
plt.rc("font", family="sans-serif")
plt.rc("font", size=14)

# import pandas_datareader.data as web
import yfinance as yf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Modeling
from arch import arch_model

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",80)

In [ ]:
yesterday = str(date.today() - timedelta(days = 1))
print("Today's date:", yesterday)

In [ ]:
start_date = "2018-01-01"
tickers = ['MAT','DIS','KO', 'NVDA','PFE', "AAPL", "META", "TSLA", "^GSPC","MSFT",]
print(f"The number of reproducible classroom price series is {len(tickers)}")

rng = np.random.default_rng(2026)
dates = pd.bdate_range(start=start_date, end=yesterday)
daily_mean = np.linspace(0.00010, 0.00045, len(tickers))
daily_volatility = np.linspace(0.009, 0.020, len(tickers))
log_returns = rng.normal(daily_mean, daily_volatility, size=(len(dates), len(tickers)))
adj_close = pd.DataFrame(
    100 * np.exp(np.cumsum(log_returns, axis=0)),
    index=dates,
    columns=tickers,
)
adj_close.index.name = "Date"
all_data = pd.concat({"Adj Close": adj_close}, axis=1)
all_data.info()

In [ ]:
adj_close_data_completed = all_data["Adj Close"].copy()
adj_close_data_completed.head()

In [ ]:
yield_data = adj_close_data_completed.pct_change().dropna()
yield_data.head()

In [ ]:
df_log = np.log(yield_data + 1)#.dropna()
df_log.head()

In [ ]:
from arch import arch_model


returns = yield_data["AAPL"]*100

In [ ]:
ax = returns.plot()
xlim = ax.set_xlim(returns.index.min(), returns.index.max())

In [ ]:
am = arch_model(returns)

res = am.fit()

In [ ]:
print(res.summary())

In [ ]:
from arch import arch_model

am = arch_model(returns)
res = am.fit(update_freq=5)
print(res.summary())

In [ ]:
fig = res.plot(annualize="D")

### ARCH Model

In [ ]:
am = arch_model(
    returns, 
    vol="ARCH",
    p = 3,
    q=0,
    o=0,
    dist = 'normal'
)

am.fit()

In [ ]:
fig = res.plot()